In [10]:
import sys
import rdflib
import pandas as pd
import matplotlib.pyplot as plt
import io

rdflib.NORMALIZE_LITERALS = False

from rdflib import Graph
from pathlib import Path

dataset_folder = Path("../data/BPDP_Dataset")

train_true = dataset_folder / "Train" / "True"
train_false = dataset_folder / "Train" / "False"
test_true = dataset_folder / "Test" / "True"
test_false = dataset_folder / "Test" / "False"

print("Dataset folders configured successfully.")

Dataset folders configured successfully.


In [11]:
import re
def format_dates(text):
    dates = re.findall(r"(\d{1,4})-(\d{1,2})-(\d{1,2})(\"\^\^xsd:date)", text)
    for year, month, day, suffix in dates:
        old = f"{year}-{month}-{day}{suffix}"
        if month == "0":
            month = "1"
        if day == "0":
            day = "1"
        proper = f"{int(year):04d}-{int(month):02d}-{int(day):02d}{suffix}"
        if old == proper: continue
        text = text.replace(old, proper)
    return text

In [12]:
# Organise the four dataset folders
dataset_groups = [
    ("Train True", train_true),
    ("Train False", train_false),
    ("Test True", test_true),
    ("Test False", test_false)
]

# Store the results for later analysis
all_nodes = set()

# Analyse each dataset group
for group_name, folder in dataset_groups:

    files = sorted(folder.glob("*.ttl"))

    print("\nDataset group:", group_name)
    print("Number of files:", len(files))

    # First, inspect one example file
    for file in files:

        graph = Graph()
        content = file.read_text()
        stream = io.StringIO(format_dates(content))
        graph.parse(stream, format="turtle")
        all_nodes = all_nodes.union(graph.all_nodes())

all_nodes

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001DE46BB65C0>
Traceback (most recent call last):
  File "c:\Users\wikid\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\Users\wikid\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdflib\xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
ValueError: Invalid isoformat string: '-0525-01-01'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001DE46BB65C0>
Traceback (most recent call last):
  File "c:\Users\wikid\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: 


Dataset group: Train True
Number of files: 100


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001DE46BB65C0>
Traceback (most recent call last):
  File "c:\Users\wikid\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\Users\wikid\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdflib\xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
ValueError: Invalid isoformat string: '-0525-01-01'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001DE46BB65C0>
Traceback (most recent call last):
  File "c:\Users\wikid\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: 


Dataset group: Train False
Number of files: 100

Dataset group: Test True
Number of files: 103

Dataset group: Test False
Number of files: 103


{rdflib.term.URIRef('http://fr.dbpedia.org/resource/Ferdinand_II_(roi_de_Portugal)'),
 rdflib.term.URIRef('http://de.dbpedia.org/resource/Valladolid'),
 rdflib.term.URIRef('http://dbpedia.org/resource/Ä°zmir'),
 rdflib.term.Literal('1869-06-27', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#date')),
 rdflib.term.Literal('Sevilla', lang='de'),
 rdflib.term.Literal('Cambridge', lang='fr'),
 rdflib.term.Literal('Albert Einstein', lang='en'),
 rdflib.term.URIRef('http://fr.dbpedia.org/resource/Orbe_(Vaud)'),
 rdflib.term.Literal('La Quinta, California', lang='en'),
 rdflib.term.Literal('Henry Moseley', lang='fr'),
 rdflib.term.Literal('1898-11-18', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#date')),
 rdflib.term.Literal('James Clavell', lang='en'),
 rdflib.term.Literal('1904-07-05', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#date')),
 rdflib.term.Literal('Alsace', lang='en'),
 rdflib.term.Literal('Sydney', lang='en'),
 rdflib.term.Liter

In [13]:
from SPARQLWrapper import SPARQLWrapper, JSON, POST

sparql = SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)
sparql.setMethod(POST)

FOAF = "http://xmlns.com/foaf/0.1/isPrimaryTopicOf"

uri_list = [node for node in all_nodes if isinstance(node, rdflib.term.URIRef)]

values = " ".join(f"<{uri}>" for uri in uri_list)

query = f"""
SELECT ?entity ?topic WHERE {{
    VALUES ?entity {{ {values} }}
    ?entity <{FOAF}> ?topic .
}}
"""

sparql.setQuery(query)

results = sparql.query().convert()

articles = {}

for result in results["results"]["bindings"]:
    entity = result["entity"]["value"]
    topic = result["topic"]["value"]
    articles[entity] = topic

articles

{'http://dbpedia.org/resource/Adam_Mickiewicz': 'http://en.wikipedia.org/wiki/Adam_Mickiewicz',
 'http://dbpedia.org/resource/Adelaide_of_Italy': 'http://en.wikipedia.org/wiki/Adelaide_of_Italy',
 'http://dbpedia.org/resource/Bing_Crosby': 'http://en.wikipedia.org/wiki/Bing_Crosby',
 'http://dbpedia.org/resource/Emma_Goldman': 'http://en.wikipedia.org/wiki/Emma_Goldman',
 'http://dbpedia.org/resource/Isaac_Bashevis_Singer': 'http://en.wikipedia.org/wiki/Isaac_Bashevis_Singer',
 'http://dbpedia.org/resource/John_A._Roebling': 'http://en.wikipedia.org/wiki/John_A._Roebling',
 'http://dbpedia.org/resource/Leoncin': 'http://en.wikipedia.org/wiki/Leoncin',
 'http://dbpedia.org/resource/Lithuania_Governorate': 'http://en.wikipedia.org/wiki/Lithuania_Governorate',
 'http://dbpedia.org/resource/Surfside,_Florida': 'http://en.wikipedia.org/wiki/Surfside,_Florida',
 'http://dbpedia.org/resource/Claude_Auchinleck': 'http://en.wikipedia.org/wiki/Claude_Auchinleck',
 'http://dbpedia.org/resource/Co

In [ ]:
import requests
import time
from urllib.parse import urlparse, unquote

session = requests.Session()
session.headers.update({
    "User-Agent": "MyResearchProject/1.0 (contact: your-email@example.com)"
})

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def get_wikipedia_intros(urls, batch_size=20):

    results = {}

    # Grupowanie URL-i według domeny/języka Wikipedii
    grouped = {}

    for url in urls:
        parsed = urlparse(str(url))
        api_url = f"{parsed.scheme}://{parsed.netloc}/w/api.php"
        title = unquote(parsed.path.split("/wiki/")[-1])

        grouped.setdefault(api_url, []).append((str(url), title))

    for api_url, articles in grouped.items():

        for batch in chunks(articles, batch_size):

            params = {
                "action": "query",
                "prop": "extracts",
                "exintro": 1,
                "explaintext": 1,
                "titles": "|".join(title for _, title in batch),
                "format": "json"
            }

            for attempt in range(5):
                try:
                    response = session.get(
                        api_url,
                        params=params,
                        timeout=30
                    )

                    if response.status_code == 429:
                        wait = int(response.headers.get("Retry-After", 10))
                        time.sleep(wait)
                        continue

                    response.raise_for_status()
                    data = response.json()

                    for page in data["query"]["pages"].values():
                        title = page["title"]
                        extract = page.get("extract", "")

                        for url, original_title in batch:
                            if original_title.replace("_", " ") == title:
                                results[url] = extract

                    break

                except requests.RequestException as e:
                    if attempt == 4:
                        print(f"Error: {e}")
                    else:
                        time.sleep(2 ** attempt)

            time.sleep(2)

    return results

In [32]:
descriptions = get_wikipedia_intros(articles.values())
descriptions

{'http://en.wikipedia.org/wiki/Adam_Mickiewicz': 'Adam Bernard Mickiewicz (24 December 1798 – 26 November 1855) was a Polish poet, dramatist, essayist, publicist, translator and political activist. He is regarded as a national poet in Poland, Lithuania and Belarus. He also largely influenced Ukrainian literature and affected Russian literature. A principal figure in Polish Romanticism, he is one of Poland\'s "Three Bards" (Polish: trzej wieszcze) and is widely regarded as Poland\'s greatest poet. He is also considered one of the greatest Slavic and European poets and has been dubbed a "Slavic bard". \nHe is known chiefly for the poetic drama Dziady (Forefathers\' Eve) and the national epic poem Pan Tadeusz. His other influential works include Konrad Wallenrod and Grażyna. All these served as inspiration for uprisings against the three imperial powers that had partitioned the Polish–Lithuanian Commonwealth out of existence. His works also promoted the Lithuanian National Revival; the op

In [33]:
for entity, url in articles.items():
    description = descriptions[url]
    filename = entity.split("/")[-1] + ".txt"
    with open(f"../data/Wikipedia/{filename}", "w", encoding="utf-8") as file:
        file.write(description)

In [23]:
print(f"All nodes: {len(uri_list)}")
print(f"Descriptions: {len(articles)}")

language = 0
alt = 0
other = []

for node in uri_list:
    if str(node) in articles: continue
    if re.search(r"[a-z]+\.dbpedia", str(node)): language += 1
    elif str(node).endswith("__1"): alt += 1
    else: other.append(node)

print("Reasons for missing")
print(f"Different language: {language}")
print(f"Alt version: {alt}")
print(f"Other: {len(other)}")
print(*other, sep="\n")

All nodes: 821
Descriptions: 229
Reasons for missing
Different language: 473
Alt version: 102
Other: 17
http://dbpedia.org/resource/Ä°zmir
http://dbpedia.org/resource/AlcalĂˇ_de_Henares
http://dbpedia.org/resource/FrĂ©dĂ©ric_Bastiat
http://dbpedia.org/resource/Eduard_von_BĂ¶hm-Ermolli
http://dbpedia.org/resource/AndrĂ©_Weil
http://dbpedia.org/resource/Isabella_II_of_Spain
http://dbpedia.org/resource/ChillĂˇn
http://dbpedia.org/resource/Lyons-la-ForĂŞt
http://dbpedia.org/resource/CĂˇdiz
http://dbpedia.org/resource/EugĂ¨ne_Viollet-le-Duc
http://dbpedia.org/resource/Constantin_BrĂ˘ncuČ™i
http://dbpedia.org/resource/SĂŁo_Paulo
http://dbpedia.org/resource/Comte_de_LautrĂ©amont
http://dbpedia.org/resource/Eleusis
http://dbpedia.org/resource/MĂĽhlhausen
http://dbpedia.org/resource/CĂłrdoba,_Spain
http://dbpedia.org/resource/Sighetu_MarmaČ›iei


In [25]:
def map_alt(ref: rdflib.term.URIRef):
    if str(ref).endswith("__1"): return str(ref)[:-3]
    return str(ref)

In [26]:
unmapped = []
for node in uri_list:
    if str(node) in articles: continue
    if str(node).endswith("__1") and map_alt(node) not in articles: unmapped.append(node)

unmapped

[rdflib.term.URIRef('http://dbpedia.org/resource/Eduard_von_BĂ¶hm-Ermolli__1'),
 rdflib.term.URIRef('http://dbpedia.org/resource/Comte_de_LautrĂ©amont__1'),
 rdflib.term.URIRef('http://dbpedia.org/resource/Constantin_BrĂ˘ncuČ™i__1'),
 rdflib.term.URIRef('http://dbpedia.org/resource/AndrĂ©_Weil__1'),
 rdflib.term.URIRef('http://dbpedia.org/resource/EugĂ¨ne_Viollet-le-Duc__1'),
 rdflib.term.URIRef('http://dbpedia.org/resource/FrĂ©dĂ©ric_Bastiat__1'),
 rdflib.term.URIRef('http://dbpedia.org/resource/Isabella_II_of_Spain__1')]